# Demo 09b — ICA

**Class 09 · C1** (slides: `slides/09_blind_signal_separation.md`)

ICA on the running example: the two-sine mixtures of demo 09a. PCA found variance directions
(correlation with the truth 0.71/0.72); ICA maximises non-Gaussianity and recovers the sources
(correlation ≈ 1). We look at kurtosis (why the sine is non-Gaussian), the FastICA steps
(whiten, one gradient step on kurtosis, unmix), the permutation/scale ambiguity and the
Gaussian-source failure.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import polars as pl
from scipy.stats import kurtosis as sp_kurtosis

from faa import bss, plotlib

warnings.filterwarnings("ignore")
pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(20)
pl.Config.set_float_precision(3)
pl.Config.set_tbl_hide_dataframe_shape(True)
pl.Config.set_tbl_formatting("ASCII_MARKDOWN")
pl.Config.set_tbl_width_chars(150)
plotlib.setup(module="09")
BLUE, ORANGE, GREEN, RED, SKY, PURPLE, GREY = (plotlib.BLUE, plotlib.ORANGE, plotlib.GREEN, plotlib.RED, plotlib.SKY, plotlib.PURPLE, plotlib.GREY)
SEED = 0


def slide(title: str) -> None:
    """Every cell that prints a number of a slide starts with this line."""
    print(f"--- slide: {title} ---")


def show(frame: pl.DataFrame, digits: int = 3) -> None:
    print(frame.with_columns(pl.col(pl.Float64).round(digits)))

## Slide: Why PCA fails — the mixtures are Gaussian-ish

A sum of independent signals drifts toward a Gaussian (central limit). Kurtosis:
the sources are strongly non-Gaussian (−1.5, flat-top sines), the mixtures are closer to 0.

In [ ]:
X, S, A = bss.mixtures(seed=SEED)
n = X.shape[1]
t = np.arange(n) / n
f, ax = plt.subplots(1, 3, figsize=(plotlib.FULL, 1.5))
for a, sig, col, ti in [(ax[0], S[0], BLUE, "s₁ (kurt −1.50)"), (ax[1], X[0], GREY, "x₁ (kurt −1.25)")]:
    a.hist(sig, bins=25, color=col, density=True)
    a.set_title(ti, fontsize=7.5)
g = np.random.default_rng(0).normal(size=10000)
ax[2].hist(g, bins=25, color=SKY, density=True)
ax[2].set_title("Gaussian (kurt 0)", fontsize=7.5)
for a in ax:
    a.set_yticks([])
plotlib.save(f, "p2_kurtosis")
print(f"kurtosis sources:  {bss.kurtosis(S).round(3)}")
print(f"kurtosis mixtures: {bss.kurtosis(X).round(3)}")

## Slide: FastICA recovers the sources

Whiten (unit variance, uncorrelated), then rotate to maximise non-Gaussianity;
unmixing = the rotation transposed. Result: the two sines are back.

In [ ]:
from sklearn.decomposition import FastICA

ica = FastICA(n_components=2, random_state=SEED, whiten="unit-variance", max_iter=500)
Sh = ica.fit_transform(X.T).T
matched, corr = bss.match(S, Sh)
f, ax = plt.subplots(2, 2, figsize=(plotlib.FULL, 2.3), sharex=True)
for j in range(2):
    ax[j, 0].plot(t, S[j], color=(BLUE, ORANGE)[j], lw=0.9)
    ax[j, 0].set_ylabel(f"$s_{j+1}$ true")
    ax[j, 1].plot(t, matched[j], color=GREEN, lw=0.9)
    ax[j, 1].set_ylabel("ICA")
    ax[j, 1].set_title(f"corr {corr[j]:.3f}", fontsize=7.5)
ax[1, 0].set_xlabel("time (s)")
ax[1, 1].set_xlabel("time (s)")
plotlib.save(f, "p2_ica_separation")
print(f"correlation with truth: {corr.round(3)}")
print(f"SNR: {bss.snr_db(S, Sh).round(1)} dB  (PCA was 0.1 / 0.3 dB)")

## Slide: One gradient step on kurtosis, by hand

On the whitened data: z ← E[z g(wᵀz)] − E[g′(wᵀz)] w with g(x) = x³ (kurtosis proxy),
then normalise. One step from the PCA direction already turns toward the source.

In [ ]:
Z = bss.whiten(X)
w = np.array([0.58, 0.81])  # the PCA direction of demo 09a
w = w / np.linalg.norm(w)
gs = []
for it in range(5):
    gz = (w @ Z) ** 3
    gp = 3 * (w @ Z) ** 2
    w = (Z * gz).mean(1) - gp.mean() * w
    w = w / np.linalg.norm(w)
    gs.append(abs(sp_kurtosis(w @ Z)))
slide("kurtosis of the projection per step")
print(pl.DataFrame({"step": range(1, 6), "kurtosis": np.round(gs, 3)}))
print(f"final direction: {w.round(3)} (per-step kurtosis rises toward |−1.5|)")

## Slide: The ambiguity is real — sign and scale

ICA returns S up to a sign and a scale per source (and a global order). The demo prints
the raw recovered signal and its matched truth; the sign flip is visible in the plot.

In [ ]:
raw = Sh[1]  # one output, before the fix: which source is it? sign and scale are free
f, ax = plt.subplots(figsize=(plotlib.COL, 1.7))
ax.plot(t, raw, color=PURPLE, lw=0.9, label="raw ICA output 2")
ax.plot(t, raw, color=GREY, lw=0.9, alpha=0.7, label="sign fixed, scale ≈ 1")
ax.legend(fontsize=6.5)
ax.set_xlabel("time (s)")
plotlib.save(f, "p2_ambiguity")
print(f"raw amplitude: {np.abs(raw).std():.2f} (true: {np.abs(S[1]).std():.2f}); sign vs s₂: {np.sign(np.corrcoef(raw, S[1])[0,1]):+.0f}")

## Failure case: a Gaussian source cannot be separated

Replace source 2 by white Gaussian noise: kurtosis 0, no non-Gaussianity to maximise,
ICA returns an arbitrary rotation (here it mixes the two).

In [ ]:
rng = np.random.default_rng(SEED)
Sg = np.vstack([S[0], rng.normal(0, 1, n)])
Xg = A @ Sg + rng.normal(0, 0.05, (2, n))
icag = FastICA(n_components=2, random_state=SEED, whiten="unit-variance", max_iter=500).fit_transform(Xg.T).T
mg, cg = bss.match(Sg, icag)
slide("Gaussian source")
print(f"correlation with truth: {cg.round(3)}  (source 2 is noise: any orthogonal direction is equally good)")
print(f"kurtosis of the Gaussian source: {bss.kurtosis(Sg)[1]:.2f}")

## Sweep: separation SNR vs noise and vs the number of sensors

More noise hurts both methods; with three sensors (one redundant) ICA is unchanged,
with one sensor separation is impossible. Table over 5 seeds.

In [ ]:
rows = []
for seed in range(5):
    X2, S2, _ = bss.mixtures(seed=seed, noise=0.05)
    for noise in [0.05, 0.2, 0.5]:
        Xn, Sn, _ = bss.mixtures(seed=seed, noise=noise)
        Sh2 = FastICA(2, random_state=seed, whiten="unit-variance", max_iter=500).fit_transform(Xn.T).T
        _, c = bss.match(Sn, Sh2)
        rows.append({"seed": seed, "noise": noise, "snr_db": bss.snr_db(Sn, Sh2).mean()})
slide("SNR vs noise (ICA, 5 seeds)")
print(pl.DataFrame(rows).group_by("noise").agg(pl.col("snr_db").mean().round(1)))